In [1]:
from langchain_aws import BedrockEmbeddings
from langchain_chroma import Chroma

In [2]:
embeddings = BedrockEmbeddings(model_id='amazon.titan-embed-text-v2:0')

vector_store = Chroma(
    collection_name="docs_collection",
    embedding_function=embeddings,
    persist_directory="./data",
)

In [3]:
uploaded_file_paths = [('C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf', 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf'), ('C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\54e651a1ec8ad98405b81b55a14bc6d4ba9fc0937a66ad97706a75aeb764386b\\132065Compliance Pack.pdf', 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\54e651a1ec8ad98405b81b55a14bc6d4ba9fc0937a66ad97706a75aeb764386b\\132065Compliance Pack.pdf'), ('C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\e7cabeafcc0ace69d083efcb36d483f884f5b85593ef964a105b87b5972d1280\\132065GA MV-1.pdf', 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\e7cabeafcc0ace69d083efcb36d483f884f5b85593ef964a105b87b5972d1280\\132065GA MV-1.pdf')]

In [7]:
file_paths = [tuple[0] for tuple in uploaded_file_paths]
file_paths

['C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf',
 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\54e651a1ec8ad98405b81b55a14bc6d4ba9fc0937a66ad97706a75aeb764386b\\132065Compliance Pack.pdf',
 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\e7cabeafcc0ace69d083efcb36d483f884f5b85593ef964a105b87b5972d1280\\132065GA MV-1.pdf']

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)

In [6]:
from langchain_core.documents import Document

In [8]:
import importlib.util
import sys
from pathlib import Path

module_path = Path("..") / "core" / "ocr_engine.py"
spec = importlib.util.spec_from_file_location("ocr_engine", module_path)
ocr_engine = importlib.util.module_from_spec(spec)
sys.modules["ocr_engine"] = ocr_engine
spec.loader.exec_module(ocr_engine)

In [9]:
# storing each_docs's pagae's text content in langchain document object
from uuid import uuid4
import json

ocr = ocr_engine.OCREngine()

docs = []

for file_path in file_paths:
    page_text, _ = ocr.extract_text(file_path, "pdf")
    
    docs.append(Document(
        page_content=page_text,
        metadata={
            "type": "text",
            "source": file_path,
            "bounding_box": json.dumps({}),
            "page": 0,
            "key": "",
            "value": "",
            "type-value": ""
        },
        id = str(uuid4())
    ))

In [10]:
docs

[Document(id='893d8ac3-1ade-4781-98d8-200cf6932ef8', metadata={'type': 'text', 'source': 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf', 'bounding_box': '{}', 'page': 0, 'key': '', 'value': '', 'type-value': ''}, page_content='T-7 (Revised 9-2022)\nDEPARTMENT CONSTITUTION JUSTICE OF TRIN REVENUE\nWeb and MV Manual\nGeorgia Department of Revenue - Motor Vehicle Division\nBill of Sale\nGEORGIA\nANY CORRECTION OR ALTERATION WILL VOID THIS FORM\nPurpose of this form: This form is to be used to provide evidence that a transaction between the purchaser(s)/transferee(s) and seller(s)/transferor(s) has taken\nplace and that the odometer reading has been declared by the vehicle\'s seller(s) and acknowledged by the vehicle\'s buyer(s).\nCompleting this form: This form must be completed in its entirety, legibly printed or typed.\nSection A: Record the vehicle\'s information. Please note: Federal regul

In [11]:
docs_splits = text_splitter.split_documents(docs)

In [12]:
docs_splits

[Document(metadata={'type': 'text', 'source': 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf', 'bounding_box': '{}', 'page': 0, 'key': '', 'value': '', 'type-value': ''}, page_content="T-7 (Revised 9-2022)\nDEPARTMENT CONSTITUTION JUSTICE OF TRIN REVENUE\nWeb and MV Manual\nGeorgia Department of Revenue - Motor Vehicle Division\nBill of Sale\nGEORGIA\nANY CORRECTION OR ALTERATION WILL VOID THIS FORM\nPurpose of this form: This form is to be used to provide evidence that a transaction between the purchaser(s)/transferee(s) and seller(s)/transferor(s) has taken\nplace and that the odometer reading has been declared by the vehicle's seller(s) and acknowledged by the vehicle's buyer(s).\nCompleting this form: This form must be completed in its entirety, legibly printed or typed.\nSection A: Record the vehicle's information. Please note: Federal regulations require the seller(s)/transferor(s) to 

In [13]:
def func(doc):
    doc.id = str(uuid4())
    return doc

docs_splits = list(map(func, docs_splits))

In [14]:
docs_splits

[Document(id='d5f9d881-200f-4d16-b31a-2b49fb7d04d5', metadata={'type': 'text', 'source': 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf', 'bounding_box': '{}', 'page': 0, 'key': '', 'value': '', 'type-value': ''}, page_content="T-7 (Revised 9-2022)\nDEPARTMENT CONSTITUTION JUSTICE OF TRIN REVENUE\nWeb and MV Manual\nGeorgia Department of Revenue - Motor Vehicle Division\nBill of Sale\nGEORGIA\nANY CORRECTION OR ALTERATION WILL VOID THIS FORM\nPurpose of this form: This form is to be used to provide evidence that a transaction between the purchaser(s)/transferee(s) and seller(s)/transferor(s) has taken\nplace and that the odometer reading has been declared by the vehicle's seller(s) and acknowledged by the vehicle's buyer(s).\nCompleting this form: This form must be completed in its entirety, legibly printed or typed.\nSection A: Record the vehicle's information. Please note: Federal regulati

In [15]:
extracted_data_mv1 = [{'OwnersInformation': {'Owner1PhoneNumber': {'success': True, 'confidence': 0.9296875, 'geometry': [{'boundingBox': {'top': 0.426406059433624, 'left': 0.8098988660673117, 'width': 0.11039919400879128, 'height': 0.011722742383910578}, 'vertices': [{'x': 0.809899177174848, 'y': 0.4264062679958325}, {'x': 0.9202980600761029, 'y': 0.426406059433624}, {'x': 0.9202977106493677, 'y': 0.43812862305594946}, {'x': 0.8098988660673117, 'y': 0.43812880181753455}], 'page': 1}], 'type': 'string', 'value': '(404) 555-1212'}, 'Owner1SignatureDate': {'success': True, 'confidence': 0.91796875, 'type': 'string', 'value': ''}, 'Owner1DLNumber': {'success': True, 'confidence': 0.94140625, 'geometry': [{'boundingBox': {'top': 0.41175311102858636, 'left': 0.695593062457971, 'width': 0.10453671375091333, 'height': 0.01025743220073988}, 'vertices': [{'x': 0.6955932999614641, 'y': 0.41175334378835543}, {'x': 0.8001297762088844, 'y': 0.41175311102858636}, {'x': 0.8001295069567229, 'y': 0.42201033516047204}, {'x': 0.695593062457971, 'y': 0.42201054322932624}], 'page': 1}], 'type': 'string', 'value': 'GA123456789'}, 'Owner2DLNumber': {'success': True, 'confidence': 0.9375, 'type': 'string', 'value': ''}, 'Owner1State': {'success': True, 'confidence': 0.9375, 'geometry': [{'boundingBox': {'top': 0.4119970280301108, 'left': 0.9134593050602168, 'width': 0.022959429227554673, 'height': 0.010013081559256787}, 'vertices': [{'x': 0.9134596015015616, 'y': 0.4119970790215492}, {'x': 0.9364187342877714, 'y': 0.4119970280301108}, {'x': 0.9364184310394972, 'y': 0.42201006389162976}, {'x': 0.9134593050602168, 'y': 0.4220101095893676}], 'page': 1}], 'type': 'string', 'value': 'GA'}, 'Owner2Name': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}, 'NumOfOwners': {'success': True, 'confidence': 0.9140625, 'geometry': [{'boundingBox': {'top': 0.31748667214617704, 'left': 0.19697659934193984, 'width': 0.0057396875122559254, 'height': 0.009524511593516638}, 'vertices': [{'x': 0.19697667926546147, 'y': 0.31748669738511326}, {'x': 0.20271628685419577, 'y': 0.31748667214617704}, {'x': 0.20271620531205214, 'y': 0.3270111597595973}, {'x': 0.19697659934193984, 'y': 0.3270111837396937}], 'page': 1}], 'type': 'string', 'value': '1'}, 'Owner1Name': {'success': True, 'confidence': 0.82421875, 'geometry': [{'boundingBox': {'top': 0.4124869968512024, 'left': 0.15545540855987217, 'width': 0.08951327420739519, 'height': 0.009768837386417428}, 'vertices': [{'x': 0.15545547852293118, 'y': 0.4124871946501437}, {'x': 0.24496868276726735, 'y': 0.4124869968512024}, {'x': 0.24496858691346585, 'y': 0.4222556565743962}, {'x': 0.15545540855987217, 'y': 0.4222558342376198}], 'page': 1}], 'type': 'string', 'value': 'John A. Doe'}, 'Owner1BusinessAddress': {'success': True, 'confidence': 0.640625, 'geometry': [{'boundingBox': {'top': 0.4686565640444264, 'left': 0.1561877013690245, 'width': 0.2514435709334446, 'height': 0.011966773506560335}, 'vertices': [{'x': 0.15618778733309308, 'y': 0.468656794436669}, {'x': 0.4076312723024691, 'y': 0.4686565640444264}, {'x': 0.4076310972472102, 'y': 0.4806231764461442}, {'x': 0.1561877013690245, 'y': 0.48062333755098674}], 'page': 1}], 'type': 'string', 'value': '123 Oak Street, Atlanta, GA030303'}, 'Owner1Signature': {'success': True, 'confidence': 0.88671875, 'geometry': [{'boundingBox': {'top': 0.4745177763729282, 'left': 0.32043726546113055, 'width': 0.07009714025933411, 'height': 0.03980724017654941}, 'vertices': [{'x': 0.32043774501604416, 'y': 0.4745178311402253}, {'x': 0.39053440572046466, 'y': 0.4745177763729282}, {'x': 0.3905338435457271, 'y': 0.5143250165494776}, {'x': 0.32043726546113055, 'y': 0.5143250070625349}], 'page': 1}], 'type': 'boolean', 'value': True}}, 'LesseeInormation': {'LesseeGACounty': {'success': True, 'confidence': 0.9296875, 'type': 'string', 'value': ''}, 'DLNumber': {'success': True, 'confidence': 0.92578125, 'type': 'string', 'value': ''}, 'LesseeAddress': {'success': True, 'confidence': 0.92578125, 'type': 'string', 'value': ''}, 'LesseeName': {'success': True, 'confidence': 0.92578125, 'type': 'string', 'value': ''}, 'LesseePhoneNumber': {'success': True, 'confidence': 0.93359375, 'type': 'string', 'value': ''}}, 'Signature': {'success': True, 'confidence': 0.87890625, 'type': 'boolean', 'value': ''}, 'VehicleInformation': {'VechicleOdometerReading': {'success': True, 'confidence': 0.93359375, 'geometry': [{'boundingBox': {'top': 0.211984165298925, 'left': 0.1767056308049378, 'width': 0.04518436191164704, 'height': 0.009891178284711255}, 'vertices': [{'x': 0.17670570786621542, 'y': 0.21198447376165033}, {'x': 0.22188999271658486, 'y': 0.211984165298925}, {'x': 0.22188990242279194, 'y': 0.2218750454122097}, {'x': 0.1767056308049378, 'y': 0.22187534358363625}], 'page': 1}], 'type': 'string', 'value': '25500'}, 'VehicleModel': {'success': True, 'confidence': 0.87890625, 'geometry': [{'boundingBox': {'top': 0.1698564462398099, 'left': 0.15264838103984152, 'width': 0.047626772653979826, 'height': 0.011967125216720415}, 'vertices': [{'x': 0.15264846575061822, 'y': 0.16985681757862572}, {'x': 0.20027515369382135, 'y': 0.1698564462398099}, {'x': 0.2002750521079395, 'y': 0.18182321324200343}, {'x': 0.15264838103984152, 'y': 0.1818235714565303}], 'page': 1}], 'type': 'string', 'value': 'Camry'}, 'IsLeasedVechicle': {'success': True, 'confidence': 0.890625, 'geometry': [{'boundingBox': {'top': 0.31797399450344754, 'left': 0.41862416892918536, 'width': 0.03785755350377101, 'height': 0.010257357029843372}, 'vertices': [{'x': 0.4186243223158731, 'y': 0.3179741605490502}, {'x': 0.45648172243295637, 'y': 0.31797399450344754}, {'x': 0.45648155754876796, 'y': 0.32823119442946314}, {'x': 0.41862416892918536, 'y': 0.3282313515332909}], 'page': 1}, {'boundingBox': {'top': 0.31895065034092396, 'left': 0.4816384435009804, 'width': 0.02540131195371187, 'height': 0.009036220575431353}, 'vertices': [{'x': 0.48163859548683297, 'y': 0.318950761181062}, {'x': 0.5070397554546923, 'y': 0.31895065034092396}, {'x': 0.5070395966727393, 'y': 0.32798676536163013}, {'x': 0.4816384435009804, 'y': 0.3279868709163553}], 'page': 1}], 'type': 'boolean', 'value': False}, 'VehicleColor': {'success': True, 'confidence': 0.90625, 'geometry': [{'boundingBox': {'top': 0.15434289216246466, 'left': 0.8563130267249113, 'width': 0.029309968274701892, 'height': 0.010257632744074707}, 'vertices': [{'x': 0.8563133130427636, 'y': 0.15434313115278603}, {'x': 0.8856229949996132, 'y': 0.15434289216246466}, {'x': 0.8856226997800285, 'y': 0.16460029283915473}, {'x': 0.8563130267249113, 'y': 0.16460052490653937}], 'page': 1}], 'type': 'string', 'value': 'Red'}, 'VehicleFuelType': {'success': True, 'confidence': 0.9140625, 'geometry': [{'boundingBox': {'top': 0.18487060089988708, 'left': 0.8558236910356135, 'width': 0.06350457802776543, 'height': 0.009891513548608372}, 'vertices': [{'x': 0.8558239669842691, 'y': 0.18487107407071235}, {'x': 0.9193282690633789, 'y': 0.18487060089988708}, {'x': 0.919327974516466, 'y': 0.19476165574162208}, {'x': 0.8558236910356135, 'y': 0.19476211444849545}], 'page': 1}], 'type': 'string', 'value': 'Gasoline'}, 'CurrentTitle': {'success': True, 'confidence': 0.828125, 'geometry': [{'boundingBox': {'top': 0.1409127924208972, 'left': 0.5270706684317106, 'width': 0.09623236281933645, 'height': 0.009891805687370275}, 'vertices': [{'x': 0.5270708481008253, 'y': 0.14091360686061377}, {'x': 0.623303031251047, 'y': 0.1409127924208972}, {'x': 0.6233028233991567, 'y': 0.15080380558679982}, {'x': 0.5270706684317106, 'y': 0.15080459810826746}], 'page': 1}], 'type': 'string', 'value': 'GA11223344'}, 'CurrentTitleState': {'success': True, 'confidence': 0.62109375, 'type': 'string', 'value': ''}, 'VehicleManufactureYear': {'success': True, 'confidence': 0.91015625, 'geometry': [{'boundingBox': {'top': 0.14139898748171173, 'left': 0.8528939376294828, 'width': 0.03663739199059801, 'height': 0.009891374830499539}, 'vertices': [{'x': 0.8528942127204682, 'y': 0.14139929713974045}, {'x': 0.8895313296200809, 'y': 0.14139898748171173}, {'x': 0.8895310437993202, 'y': 0.15129006099880685}, {'x': 0.8528939376294828, 'y': 0.15129036231221127}], 'page': 1}], 'type': 'string', 'value': '2020'}, 'VIN': {'success': True, 'confidence': 0.85546875, 'geometry': [{'boundingBox': {'top': 0.1399385071631468, 'left': 0.15240436648368155, 'width': 0.1638853081478367, 'height': 0.010136517017029667}, 'vertices': [{'x': 0.15240443815536084, 'y': 0.13993989785605854}, {'x': 0.31628967463151825, 'y': 0.1399385071631468}, {'x': 0.31628955377972606, 'y': 0.15007367173612107}, {'x': 0.15240436648368155, 'y': 0.15007502418017646}], 'page': 1}], 'type': 'string', 'value': '1NXBR32E36Z754321'}, 'VehicleManufacturer': {'success': True, 'confidence': 0.85546875, 'geometry': [{'boundingBox': {'top': 0.15581379472824392, 'left': 0.15130517105492935, 'width': 0.04970282124355785, 'height': 0.011112398519258104}, 'vertices': [{'x': 0.15130524927303726, 'y': 0.1558141983259175}, {'x': 0.2010079922984872, 'y': 0.15581379472824392}, {'x': 0.20100789772758523, 'y': 0.16692580236791105}, {'x': 0.15130517105492935, 'y': 0.16692619324750202}], 'page': 1}], 'type': 'string', 'value': 'Toyota'}, 'VehiclePurchaseDate': {'success': True, 'confidence': 0.8515625, 'geometry': [{'boundingBox': {'top': 0.21149294168196633, 'left': 0.5671252699960873, 'width': 0.06203813847037576, 'height': 0.010135604385062946}, 'vertices': [{'x': 0.5671254661211775, 'y': 0.21149336589850584}, {'x': 0.629163408466463, 'y': 0.21149294168196633}, {'x': 0.6291631937242128, 'y': 0.22162813632931236}, {'x': 0.5671252699960873, 'y': 0.22162854606702928}], 'page': 1}], 'type': 'string', 'value': '1/8/2025'}}}]

In [16]:
extracted_data_bos = [{'Car-Year': {'success': True, 'confidence': 0.94140625, 'geometry': [{'boundingBox': {'top': 0.4178521801927104, 'left': 0.08523806697669341, 'width': 0.03040760714175332, 'height': 0.008547688225081063}, 'vertices': [{'x': 0.08523809000365389, 'y': 0.4178521801927104}, {'x': 0.11564567411844673, 'y': 0.41785232631541697}, {'x': 0.1156456492726704, 'y': 0.4263998684177915}, {'x': 0.08523806697669341, 'y': 0.426399721897232}], 'page': 1}], 'type': 'string', 'value': '2021'}, 'Car-VIN': {'success': True, 'confidence': 0.73828125, 'geometry': [{'boundingBox': {'top': 0.38929036259640587, 'left': 0.26906356215476057, 'width': 0.6734135072744424, 'height': 0.009389162063704548}, 'vertices': [{'x': 0.26906359951430603, 'y': 0.38929036259640587}, {'x': 0.942477069429203, 'y': 0.3892935692205155}, {'x': 0.9424769878387455, 'y': 0.3986795246601104}, {'x': 0.26906356215476057, 'y': 0.39867630836081935}], 'page': 1}], 'type': 'string', 'value': 'INXBR32E36Z754321'}, 'Customer2-Name': {'success': True, 'confidence': 0.9453125, 'type': 'string', 'value': ''}, 'Trade-Car-Year': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}, 'Dealer-Signature': {'success': True, 'confidence': 0.703125, 'type': 'boolean', 'value': False}, 'Customer-ID': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}, 'Trade-Car-Mileage': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}, 'Trade-Car-Make': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}, 'Policy-Number': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}, 'Lien-Holder-Name': {'success': True, 'confidence': 0.9140625, 'geometry': [{'boundingBox': {'top': 0.750229356101389, 'left': 0.21028654319537848, 'width': 0.12407253123660222, 'height': 0.011722951941725568}, 'vertices': [{'x': 0.21028658503301376, 'y': 0.750229356101389}, {'x': 0.3343590744319807, 'y': 0.7502300154548404}, {'x': 0.33435902241652654, 'y': 0.7619523080431145}, {'x': 0.21028654319537848, 'y': 0.7619516464633501}], 'page': 1}], 'type': 'string', 'value': 'Georgia Auto Bank'}, 'Trade-Car-Model': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}, 'Car-Model': {'success': True, 'confidence': 0.8984375, 'geometry': [{'boundingBox': {'top': 0.41736639778479134, 'left': 0.6364827735412659, 'width': 0.042985970439566534, 'height': 0.011478344250709871}, 'vertices': [{'x': 0.6364828487405161, 'y': 0.41736639778479134}, {'x': 0.6794687439808325, 'y': 0.4173666043200442}, {'x': 0.6794686653288465, 'y': 0.4288447420355012}, {'x': 0.6364827735412659, 'y': 0.4288445347449887}], 'page': 1}], 'type': 'string', 'value': 'Camry'}, 'County': {'success': True, 'confidence': 0.92578125, 'type': 'string', 'value': ''}, 'Car-Price ': {'success': True, 'confidence': 0.86328125, 'geometry': [{'boundingBox': {'top': 0.44642884180762565, 'left': 0.7976796812350837, 'width': 0.04445141712907286, 'height': 0.010501488900187927}, 'vertices': [{'x': 0.797679761880192, 'y': 0.44642884180762565}, {'x': 0.8421310983641566, 'y': 0.44642905736140315}, {'x': 0.8421310144524714, 'y': 0.4569303307078136}, {'x': 0.7976796812350837, 'y': 0.45693011443949766}], 'page': 1}], 'type': 'string', 'value': '18500'}, 'Customer-Address': {'State': {'success': True, 'confidence': 0.90625, 'geometry': [{'boundingBox': {'top': 0.697479907644002, 'left': 0.3788104620928774, 'width': 0.021248692204160036, 'height': 0.008791838201490876}, 'vertices': [{'x': 0.37881050383928905, 'y': 0.697479907644002}, {'x': 0.4000591542970374, 'y': 0.6974800188490576}, {'x': 0.4000591112433351, 'y': 0.7062717458454929}, {'x': 0.3788104620928774, 'y': 0.7062716343544782}], 'page': 1}], 'type': 'string', 'value': 'GA'}, 'Street-Address ': {'success': True, 'confidence': 0.79296875, 'geometry': [{'boundingBox': {'top': 0.6711039917159255, 'left': 0.2383741200605169, 'width': 0.07082885780029097, 'height': 0.009280526201123962}, 'vertices': [{'x': 0.23837415500604367, 'y': 0.6711039917159255}, {'x': 0.30920297786080786, 'y': 0.6711043595397227}, {'x': 0.3092029383155562, 'y': 0.6803845179170495}, {'x': 0.2383741200605169, 'y': 0.6803841490870998}], 'page': 1}], 'type': 'string', 'value': 'Oak Street'}, 'Zip-Code': {'success': True, 'confidence': 0.9296875, 'geometry': [{'boundingBox': {'top': 0.6979691589464982, 'left': 0.535855105718151, 'width': 0.04054347398323266, 'height': 0.008791941524247449}, 'vertices': [{'x': 0.5358551571264965, 'y': 0.6979691589464982}, {'x': 0.5763985797013836, 'y': 0.697969371161278}, {'x': 0.576398525798665, 'y': 0.7067611004707457}, {'x': 0.535855105718151, 'y': 0.7067608877103426}], 'page': 1}], 'type': 'string', 'value': '30303'}, 'City': {'success': True, 'confidence': 0.91796875, 'geometry': [{'boundingBox': {'top': 0.6979668002960497, 'left': 0.08523731300973848, 'width': 0.045428132319563094, 'height': 0.008303532048954398}, 'vertices': [{'x': 0.08523733537869811, 'y': 0.6979668002960497}, {'x': 0.13066544532930158, 'y': 0.6979670380785491}, {'x': 0.1306654203207213, 'y': 0.7062703323450041}, {'x': 0.08523731300973848, 'y': 0.7062700939851092}], 'page': 1}], 'type': 'string', 'value': 'Atlanta'}}, 'Trade-Car-VIN': {'success': True, 'confidence': 0.93359375, 'type': 'string', 'value': ''}, 'Car-Make': {'success': True, 'confidence': 0.890625, 'geometry': [{'boundingBox': {'top': 0.41760866767798593, 'left': 0.23153641243185913, 'width': 0.04469554701440248, 'height': 0.01098991430611157}, 'vertices': [{'x': 0.23153645328894298, 'y': 0.41760866767798593}, {'x': 0.2762319594462616, 'y': 0.4176088824441519}, {'x': 0.2762319151518943, 'y': 0.4285985819840975}, {'x': 0.23153641243185913, 'y': 0.42859836646605104}], 'page': 1}], 'type': 'string', 'value': 'Toyota'}, 'Deal-ID': {'success': True, 'confidence': 0.94921875, 'type': 'string', 'value': ''}, 'Car-Mileage': {'success': True, 'confidence': 0.80078125, 'type': 'string', 'value': ''}, 'Purchaser-Signature': {'success': True, 'confidence': 0.89453125, 'geometry': [{'boundingBox': {'top': 0.9187372435157589, 'left': 0.2369076270431374, 'width': 0.06765379407342748, 'height': 0.046400995667351896}, 'vertices': [{'x': 0.23690780129428857, 'y': 0.9187372435157589}, {'x': 0.30456142111656487, 'y': 0.9187376204956506}, {'x': 0.3045612248978444, 'y': 0.9651382391831108}, {'x': 0.2369076270431374, 'y': 0.9651378573980094}], 'page': 1}], 'type': 'boolean', 'value': True}, 'Customer-Name': {'success': True, 'confidence': 0.90625, 'geometry': [{'boundingBox': {'top': 0.641798190362164, 'left': 0.23190194128868277, 'width': 0.0812089598233954, 'height': 0.0092805803251923}, 'vertices': [{'x': 0.2319019758139041, 'y': 0.641798190362164}, {'x': 0.3131109011120782, 'y': 0.6417986084482034}, {'x': 0.3131108613130322, 'y': 0.6510787706873563}, {'x': 0.23190194128868277, 'y': 0.6510783514477103}], 'page': 1}], 'type': 'string', 'value': 'John A. Doe'}, 'Insurance-Company': {'success': True, 'confidence': 0.94140625, 'type': 'string', 'value': ''}}]

In [17]:
extracted_data_dl = [{'DLInformation': {'DLEndorsements': [{'confidence': 0.76953125, 'geometry': [{'boundingBox': {'top': 0.3375743328612481, 'left': 0.8239851383872213, 'width': 0.08840686202400017, 'height': 0.035150718824064286}, 'vertices': [{'x': 0.8239862142214593, 'y': 0.3375750278465643}, {'x': 0.9123920004112215, 'y': 0.3375743328612481}, {'x': 0.9123908268822993, 'y': 0.3727243284227789}, {'x': 0.8239851383872213, 'y': 0.3727250516853124}], 'page': 1}], 'type': 'string', 'value': 'NONE'}], 'DLHolderAddress': {'success': True, 'confidence': 0.71484375, 'geometry': [{'boundingBox': {'top': 0.517231772161244, 'left': 0.32309559047653463, 'width': 0.3025832459903081, 'height': 0.037105424025786204}, 'vertices': [{'x': 0.3230961418028264, 'y': 0.5172346455392157}, {'x': 0.6256788364668427, 'y': 0.517231772161244}, {'x': 0.6256779321882181, 'y': 0.5543342206507373}, {'x': 0.32309559047653463, 'y': 0.5543371961870303}], 'page': 1}, {'boundingBox': {'top': 0.5670280232589486, 'left': 0.32162957345354426, 'width': 0.21564243960617013, 'height': 0.03612820170231745}, 'vertices': [{'x': 0.3216301086053481, 'y': 0.5670301687476509}, {'x': 0.5372720130597144, 'y': 0.5670280232589486}, {'x': 0.5372712329886469, 'y': 0.60315400858355}, {'x': 0.32162957345354426, 'y': 0.6031562249612661}], 'page': 1}, {'boundingBox': {'top': 0.5675146473302547, 'left': 0.5524125181675962, 'width': 0.14213345158317037, 'height': 0.035639205630271786}, 'vertices': [{'x': 0.552413304661736, 'y': 0.5675160620839886}, {'x': 0.6945459697507665, 'y': 0.5675146473302547}, {'x': 0.6945450240088813, 'y': 0.6031523921142348}, {'x': 0.5524125181675962, 'y': 0.6031538529605265}], 'page': 1}], 'type': 'string', 'value': '123 SAMPLE STREET SACRAMENTO CA 01234'}, 'DLHolderName': {'success': True, 'confidence': 0.77734375, 'geometry': [{'boundingBox': {'top': 0.38859407080441377, 'left': 0.387082064039813, 'width': 0.11087501864285165, 'height': 0.046379446007395575}, 'vertices': [{'x': 0.3870828464986866, 'y': 0.3885949938945415}, {'x': 0.49795708268266464, 'y': 0.38859407080441377}, {'x': 0.4979561385593497, 'y': 0.43497254692928045}, {'x': 0.387082064039813, 'y': 0.43497351681180935}], 'page': 1}, {'boundingBox': {'top': 0.38883780028744686, 'left': 0.5123648725790532, 'width': 0.0297953070617456, 'height': 0.045890523743261025}, 'vertices': [{'x': 0.5123658275522861, 'y': 0.3888380484083005}, {'x': 0.5421601796407988, 'y': 0.38883780028744686}, {'x': 0.5421591816821242, 'y': 0.43472806346806364}, {'x': 0.5123648725790532, 'y': 0.4347283240307079}], 'page': 1}], 'type': 'string', 'value': 'JOHN A DOE'}, 'DLIssueDate': {'success': True, 'confidence': 0.9375, 'type': 'string', 'value': ''}, 'DLNumber': {'success': True, 'confidence': 0.95703125, 'geometry': [{'boundingBox': {'top': 0.2692288381002514, 'left': 0.38977055374299785, 'width': 0.2544747766344819, 'height': 0.04125463313000188}, 'vertices': [{'x': 0.38977125320899464, 'y': 0.269230680326232}, {'x': 0.6442453303774798, 'y': 0.2692288381002514}, {'x': 0.6442443008752454, 'y': 0.3104815334772215}, {'x': 0.38977055374299785, 'y': 0.31048347123025327}], 'page': 1}], 'type': 'string', 'value': 'GA123456789'}, 'DLRestrictions': [{'confidence': 0.78515625, 'geometry': [{'boundingBox': {'top': 0.6661306426909038, 'left': 0.4073473480496549, 'width': 0.08840615877974928, 'height': 0.03515039024758948}, 'vertices': [{'x': 0.4073479634605086, 'y': 0.6661316019920512}, {'x': 0.49575350682940417, 'y': 0.6661306426909038}, {'x': 0.4957527937244032, 'y': 0.7012800453610827}, {'x': 0.4073473480496549, 'y': 0.7012810329384933}], 'page': 1}], 'type': 'string', 'value': 'NONE'}], 'DLExpDate': {'success': True, 'confidence': 0.8828125, 'geometry': [{'boundingBox': {'top': 0.32854513695721255, 'left': 0.3931884450639607, 'width': 0.20905024665176336, 'height': 0.04930949789831529}, 'vertices': [{'x': 0.3931892864088795, 'y': 0.3285467631781128}, {'x': 0.6022386917157241, 'y': 0.32854513695721255}, {'x': 0.6022375263067118, 'y': 0.3778529148365851}, {'x': 0.3931884450639607, 'y': 0.37785463485552784}], 'page': 1}], 'type': 'string', 'value': '11/03/2028'}, 'DLClass': {'success': True, 'confidence': 0.9375, 'geometry': [{'boundingBox': {'top': 0.27240090849772597, 'left': 0.7917507177683659, 'width': 0.020026711086410298, 'height': 0.03466212735938279}, 'vertices': [{'x': 0.7917517435358986, 'y': 0.27240105404887865}, {'x': 0.8117774288547762, 'y': 0.27240090849772597}, {'x': 0.8117763812647235, 'y': 0.307062883989523}, {'x': 0.7917507177683659, 'y': 0.30706303585710876}], 'page': 1}], 'type': 'string', 'value': 'C'}, 'DLHolderHeight': {'success': True, 'confidence': 0.87890625, 'geometry': [{'boundingBox': {'top': 0.8145390638484368, 'left': 0.4014836459199432, 'width': 0.0647170845412019, 'height': 0.03466171298668008}, 'vertices': [{'x': 0.4014842463908164, 'y': 0.8145398534966902}, {'x': 0.4662007304611451, 'y': 0.8145390638484368}, {'x': 0.4662000594677334, 'y': 0.8491999667752554}, {'x': 0.4014836459199432, 'y': 0.8492007768351169}], 'page': 1}], 'type': 'string', 'value': "5'06"}, 'DLHolderDOB': {'success': True, 'confidence': 0.6953125, 'geometry': [{'boundingBox': {'top': 0.609011733851278, 'left': 0.3216287924233637, 'width': 0.26741571401403125, 'height': 0.04686885830382359}, 'vertices': [{'x': 0.32162948667228525, 'y': 0.609014496614592}, {'x': 0.5890445064373949, 'y': 0.609011733851278}, {'x': 0.5890434181721886, 'y': 0.6558777153486725}, {'x': 0.3216287924233637, 'y': 0.6558805921551016}], 'page': 1}], 'type': 'string', 'value': '0815/06/1985'}, 'DLHolderGender': {'success': True, 'confidence': 0.90625, 'geometry': [{'boundingBox': {'top': 0.7652331098102693, 'left': 0.3936696873118487, 'width': 0.027352503384945825, 'height': 0.033684988630092505}, 'vertices': [{'x': 0.3936702625937959, 'y': 0.7652334312765725}, {'x': 0.42102219069679453, 'y': 0.7652331098102693}, {'x': 0.42102158644857524, 'y': 0.7989177685902173}, {'x': 0.3936696873118487, 'y': 0.7989180984403618}], 'page': 1}], 'type': 'string', 'value': 'M'}}}]

In [18]:
extracted_data_c = [extracted_data_mv1, extracted_data_bos, extracted_data_dl]

In [19]:
from typing import Dict, List

final_data = []
for extracted_data in extracted_data_c:
    for item in extracted_data:
        for key, value in item.items():
            if isinstance(value, Dict) and "success" in value.keys():
                final_data.append({key: value})
            else:
                for skey, svalue in value.items():
                    final_data.append({skey: svalue})

In [20]:
final_data

[{'Owner1PhoneNumber': {'success': True,
   'confidence': 0.9296875,
   'geometry': [{'boundingBox': {'top': 0.426406059433624,
      'left': 0.8098988660673117,
      'width': 0.11039919400879128,
      'height': 0.011722742383910578},
     'vertices': [{'x': 0.809899177174848, 'y': 0.4264062679958325},
      {'x': 0.9202980600761029, 'y': 0.426406059433624},
      {'x': 0.9202977106493677, 'y': 0.43812862305594946},
      {'x': 0.8098988660673117, 'y': 0.43812880181753455}],
     'page': 1}],
   'type': 'string',
   'value': '(404) 555-1212'}},
 {'Owner1SignatureDate': {'success': True,
   'confidence': 0.91796875,
   'type': 'string',
   'value': ''}},
 {'Owner1DLNumber': {'success': True,
   'confidence': 0.94140625,
   'geometry': [{'boundingBox': {'top': 0.41175311102858636,
      'left': 0.695593062457971,
      'width': 0.10453671375091333,
      'height': 0.01025743220073988},
     'vertices': [{'x': 0.6955932999614641, 'y': 0.41175334378835543},
      {'x': 0.8001297762088844

In [21]:
import json

file_path = "C:\\file_path\\file_name.pdf"

for item in final_data:
    for key, value in item.items():
        if isinstance(value, List):
            continue
        if "geometry" in value.keys() and len(value["geometry"]) > 0:
            docs_splits.append(Document(
                page_content=f"Value of {key} is {value["value"]}",
                metadata={
                    "type": "key-value",
                    "source": file_path,
                    "bounding_box": json.dumps(value["geometry"][0]["boundingBox"]),
                    "page": value["geometry"][0]["page"],
                    "key": str(key),
                    "value": value["value"],
                    "type-value": value["type"]
                },
                id=str(uuid4())
            ))

In [22]:
docs

[Document(id='893d8ac3-1ade-4781-98d8-200cf6932ef8', metadata={'type': 'text', 'source': 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf', 'bounding_box': '{}', 'page': 0, 'key': '', 'value': '', 'type-value': ''}, page_content='T-7 (Revised 9-2022)\nDEPARTMENT CONSTITUTION JUSTICE OF TRIN REVENUE\nWeb and MV Manual\nGeorgia Department of Revenue - Motor Vehicle Division\nBill of Sale\nGEORGIA\nANY CORRECTION OR ALTERATION WILL VOID THIS FORM\nPurpose of this form: This form is to be used to provide evidence that a transaction between the purchaser(s)/transferee(s) and seller(s)/transferor(s) has taken\nplace and that the odometer reading has been declared by the vehicle\'s seller(s) and acknowledged by the vehicle\'s buyer(s).\nCompleting this form: This form must be completed in its entirety, legibly printed or typed.\nSection A: Record the vehicle\'s information. Please note: Federal regul

In [23]:
docs_splits

[Document(id='d5f9d881-200f-4d16-b31a-2b49fb7d04d5', metadata={'type': 'text', 'source': 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\67b0e65276d30f9f7aa536d70ddd31e35b7b017afdfa04ee25fb44e3e9256246\\132065BILL OF SALE.pdf', 'bounding_box': '{}', 'page': 0, 'key': '', 'value': '', 'type-value': ''}, page_content="T-7 (Revised 9-2022)\nDEPARTMENT CONSTITUTION JUSTICE OF TRIN REVENUE\nWeb and MV Manual\nGeorgia Department of Revenue - Motor Vehicle Division\nBill of Sale\nGEORGIA\nANY CORRECTION OR ALTERATION WILL VOID THIS FORM\nPurpose of this form: This form is to be used to provide evidence that a transaction between the purchaser(s)/transferee(s) and seller(s)/transferor(s) has taken\nplace and that the odometer reading has been declared by the vehicle's seller(s) and acknowledged by the vehicle's buyer(s).\nCompleting this form: This form must be completed in its entirety, legibly printed or typed.\nSection A: Record the vehicle's information. Please note: Federal regulati

In [24]:
_ = vector_store.add_documents(documents=docs_splits)

In [ ]:
retrieved_docs_text = vector_store.similarity_search("Compare ", filter={"type": "text"})
retrieved_docs_visual = vector_store.similarity_search("Provide me signature from the dicuments.", filter={"type": "key-value"})

In [29]:
retrieved_docs_text

[Document(id='cb0d5e86-69f3-4a3c-a02a-b05bd9c13db0', metadata={'value': '', 'page': 0, 'source': 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\e7cabeafcc0ace69d083efcb36d483f884f5b85593ef964a105b87b5972d1280\\132065GA MV-1.pdf', 'type': 'text', 'type-value': '', 'key': '', 'bounding_box': '{}'}, page_content="All owners listed on the title must sign this form. By signing this form you are agreeing to the following:\n*Owner's signature below warrants: / do solemnly swear or affirm under criminal penalty of a felony for fraudulent use of a false or fictitious\nname or address or for making a material false statement punishable by fine up to $5,000 or by imprisonment of up to five years, or both\nthat the statements contained herein are true and accurate.\nOWNER # 1\nFor owner number one:\nIf an individual, provide the full legal name, driver's license number, state of issuance, date of birth, e-mail address, telephone\nnumber, address, and mailing address (if applicable).\nIf a b

In [30]:
retrieved_docs_visual

[Document(id='33ae5cdf-bd5d-48e2-b5ac-e5e1ebc0f18b', metadata={'page': 1, 'key': 'Purchaser-Signature', 'type': 'key-value', 'source': 'C:\\file_path\\file_name.pdf', 'value': True, 'type-value': 'boolean', 'bounding_box': '{"top": 0.9187372435157589, "left": 0.2369076270431374, "width": 0.06765379407342748, "height": 0.046400995667351896}'}, page_content='Value of Purchaser-Signature is True'),
 Document(id='2dd2c819-f06d-4178-b587-c60b5d88e349', metadata={'page': 1, 'value': True, 'key': 'Owner1Signature', 'type': 'key-value', 'bounding_box': '{"top": 0.4745177763729282, "left": 0.32043726546113055, "width": 0.07009714025933411, "height": 0.03980724017654941}', 'source': 'C:\\file_path\\file_name.pdf', 'type-value': 'boolean'}, page_content='Value of Owner1Signature is True'),
 Document(id='fdf7538b-40dd-473e-9285-3166c4d9b53d', metadata={'value': '123 SAMPLE STREET SACRAMENTO CA 01234', 'type-value': 'string', 'type': 'key-value', 'page': 1, 'source': 'C:\\file_path\\file_name.pdf',

In [ ]:
retrieved_docs_text_score = vector_store.similarity_search_with_score("Name of owner/buyer", filter={"type": "text"}, k=8)
retrieved_docs_visual_score = vector_store.similarity_search_with_score("Name of owner1/buyer", filter={"type": "key-value"}, k=8)

In [32]:
for doc, score in retrieved_docs_text_score:
    print({score: doc})

{1.2844760417938232: Document(id='b8c1cfdd-8bb7-40f2-99f4-364a29ad944f', metadata={'type': 'text', 'source': 'C:\\Users\\KaranJoshi\\AppData\\Local\\Temp\\gradio\\e7cabeafcc0ace69d083efcb36d483f884f5b85593ef964a105b87b5972d1280\\132065GA MV-1.pdf', 'page': 0, 'key': '', 'bounding_box': '{}', 'value': '', 'type-value': ''}, page_content="the statements contained herein are true and accurate.\nOWNER # 1\nFull Legal Name: John A. Doe\nDriver's License GA123456789\nState: GA\nDate of Birth: 1985-06-15 E-mail Address: john.doe@example.com\nPhone #: (404) 555-1212\nBusiness Name:\nName of Agent:\nAddress: 123 Oak Street, Atlanta, GA, 30303\nMailing Address: 123 Oak Street, Atlanta, GA030303\n*Signature of Owner 1 or Business Agent: JunA\nDate:\nOWNER 2\nFull Legal Name:\nDriver's License #:\nState:\nDate of Birth:\nE-mail Address:\nPhone #:\nBusiness Name:\nName of Agent:\nAddress:\nMailing Address:\n*Signature of Owner 2 or Business Agent:\nDate:\nC\nSELLER INFORMATION\nD\nLESSEE INFORMATIO

In [33]:
for doc, score in retrieved_docs_visual_score:
    print({score: doc})

{1.227687120437622: Document(id='2dd2c819-f06d-4178-b587-c60b5d88e349', metadata={'source': 'C:\\file_path\\file_name.pdf', 'type-value': 'boolean', 'key': 'Owner1Signature', 'value': True, 'page': 1, 'type': 'key-value', 'bounding_box': '{"top": 0.4745177763729282, "left": 0.32043726546113055, "width": 0.07009714025933411, "height": 0.03980724017654941}'}, page_content='Value of Owner1Signature is True')}
{1.242713212966919: Document(id='d7d47903-2805-4b73-98e3-484897331ae3', metadata={'bounding_box': '{"top": 0.4124869968512024, "left": 0.15545540855987217, "width": 0.08951327420739519, "height": 0.009768837386417428}', 'value': 'John A. Doe', 'source': 'C:\\file_path\\file_name.pdf', 'key': 'Owner1Name', 'type-value': 'string', 'type': 'key-value', 'page': 1}, page_content='Value of Owner1Name is John A. Doe')}
{1.2436366081237793: Document(id='4ae499b0-d267-4d9b-8670-38a025a88826', metadata={'key': 'Owner1DLNumber', 'page': 1, 'value': 'GA123456789', 'bounding_box': '{"top": 0.4117